<a href="https://colab.research.google.com/github/viduladp/nova-ai-platform/blob/main/task2_mcp_notebook_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastmcp==0.4.1 faker==24.0.0 rapidfuzz==3.6.1 jsonlines==4.0.0 groq==0.13.0 httpx==0.27.2 -q

print("✅ Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.67.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
✅ Packages installed


In [3]:
import os
import json
import random
import datetime
import jsonlines
from faker import Faker
from rapidfuzz import fuzz, process
from groq import Groq
from google.colab import userdata

# Initialize
fake = Faker()
Faker.seed(42)        # Makes data reproducible — same data every run
random.seed(42)

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# Create folders
os.makedirs("task2_mcp", exist_ok=True)
os.makedirs("data", exist_ok=True)

print("✅ Imports ready")
print(f"   Faker version: {Faker.VERSION if hasattr(Faker, 'VERSION') else 'loaded'}")

✅ Imports ready
   Faker version: loaded


In [4]:
# ── NOVA Mock Database Generator ─────────────────────────────

# 1. PRODUCT CATALOG (100 products)
categories = ["skincare", "makeup", "hair", "apparel", "footwear", "accessories"]

skincare_names = ["Vitamin C Serum", "Hydrating Moisturizer", "Retinol Night Cream",
                  "SPF 50 Sunscreen", "Niacinamide Toner", "Hyaluronic Acid Serum",
                  "AHA BHA Exfoliator", "Eye Cream", "Face Cleanser", "Sheet Mask"]

makeup_names = ["Foundation", "Concealer", "Blush", "Highlighter", "Mascara",
                "Lipstick", "Eyeshadow Palette", "Setting Spray", "Primer", "Bronzer"]

hair_names = ["Shampoo", "Conditioner", "Hair Mask", "Hair Oil", "Dry Shampoo",
              "Leave-in Conditioner", "Heat Protectant", "Scalp Serum", "Hair Mist", "Curl Cream"]

apparel_names = ["Wrap Dress", "Linen Blazer", "Cargo Pants", "Silk Blouse",
                 "Denim Jacket", "Maxi Skirt", "Crop Top", "Jumpsuit", "Trench Coat", "Knit Sweater"]

footwear_names = ["Block Heels", "Sneakers", "Ankle Boots", "Sandals",
                  "Loafers", "Platform Shoes", "Ballet Flats", "Chelsea Boots", "Mules", "Espadrilles"]

accessory_names = ["Tote Bag", "Silk Scarf", "Hoop Earrings", "Layered Necklace",
                   "Sunglasses", "Hair Clips", "Belt", "Crossbody Bag", "Watch", "Bracelet"]

category_names = {
    "skincare": skincare_names,
    "makeup": makeup_names,
    "hair": hair_names,
    "apparel": apparel_names,
    "footwear": footwear_names,
    "accessories": accessory_names
}

skin_types = ["oily", "dry", "combination", "sensitive", "normal"]
ingredients_pool = [
    "Niacinamide", "Hyaluronic Acid", "Retinol", "Vitamin C", "SPF 50",
    "AHA", "BHA", "Ceramides", "Peptides", "Collagen", "Aloe Vera",
    "Tea Tree Oil", "Salicylic Acid", "Glycolic Acid", "Zinc Oxide"
]
sizes = ["XS", "S", "M", "L", "XL", "XXL"]
shoe_sizes = ["5", "6", "6.5", "7", "7.5", "8", "8.5", "9", "10", "11"]

products = []
product_id_counter = 1001

for category in categories:
    names = category_names[category]
    for name in names:
        product = {
            "product_id": f"SKU-{product_id_counter}",
            "name": f"NOVA {name}",
            "category": category,
            "price": round(random.uniform(12.99, 149.99), 2),
            "stock": random.randint(0, 500),
            "rating": round(random.uniform(3.5, 5.0), 1),
            "description": f"Premium {name.lower()} by NOVA. {fake.sentence()}",
        }

        # Category-specific fields
        if category == "skincare":
            product["ingredients"] = random.sample(ingredients_pool, random.randint(3, 6))
            product["suitable_for"] = random.sample(skin_types, random.randint(2, 4))
            product["size_ml"] = random.choice([30, 50, 100, 150, 200])

        elif category == "makeup":
            product["shades"] = [f"Shade-{i}" for i in range(1, random.randint(5, 20))]
            product["finish"] = random.choice(["matte", "dewy", "satin", "glossy"])
            product["ingredients"] = random.sample(ingredients_pool, random.randint(2, 4))

        elif category == "hair":
            product["suitable_for"] = random.sample(
                ["straight", "wavy", "curly", "coily", "color-treated", "damaged"],
                random.randint(2, 3)
            )
            product["size_ml"] = random.choice([100, 200, 250, 300, 500])

        elif category == "apparel":
            product["available_sizes"] = sizes
            product["material"] = random.choice(["cotton", "linen", "polyester", "silk", "wool", "denim"])
            product["care"] = "Machine wash cold, tumble dry low"

        elif category == "footwear":
            product["available_sizes"] = shoe_sizes
            product["material"] = random.choice(["leather", "suede", "canvas", "synthetic"])
            product["heel_height"] = random.choice(["flat", "low", "mid", "high"])

        elif category == "accessories":
            product["material"] = random.choice(["gold-plated", "silver", "brass", "leather", "fabric"])
            product["dimensions"] = f"{random.randint(10, 40)}cm x {random.randint(5, 30)}cm"

        products.append(product)
        product_id_counter += 1

print(f"✅ Generated {len(products)} products")


# 2. CUSTOMER PROFILES (200 customers)
skin_type_list = ["oily", "dry", "combination", "sensitive", "normal"]
countries = ["US", "UK", "CA", "AU", "DE", "FR", "SG", "IN", "AE", "JP"]

customers = []
customer_ids = [f"CUST-{i:04d}" for i in range(1, 201)]
product_ids = [p["product_id"] for p in products]

for cust_id in customer_ids:
    purchased = random.sample(product_ids, random.randint(1, 8))
    customer = {
        "customer_id": cust_id,
        "name": fake.name(),
        "email": fake.email(),
        "country": random.choice(countries),
        "skin_type": random.choice(skin_type_list),
        "member_since": fake.date_between(start_date="-4y", end_date="-1m").isoformat(),
        "total_orders": random.randint(1, 25),
        "total_spent": round(random.uniform(50, 2000), 2),
        "purchase_history": purchased,
        "loyalty_tier": random.choice(["bronze", "silver", "gold", "platinum"]),
        "preferences": random.sample(categories, random.randint(2, 4))
    }
    customers.append(customer)

print(f"✅ Generated {len(customers)} customers")


# 3. ORDERS (500 orders)
statuses = ["processing", "shipped", "out_for_delivery", "delivered", "cancelled", "returned"]
carriers = ["FedEx", "UPS", "DHL", "USPS", "BlueDart"]
return_reasons = ["wrong_size", "damaged", "not_as_described", "changed_mind", "defective"]

orders = []
for i in range(1, 501):
    customer = random.choice(customers)
    ordered_products = random.sample(product_ids, random.randint(1, 4))
    order_date = fake.date_between(start_date="-6m", end_date="today")
    status = random.choice(statuses)

    order = {
        "order_id": f"TRX-{1000 + i}",
        "customer_id": customer["customer_id"],
        "products": ordered_products,
        "total_amount": round(sum(random.uniform(20, 150) for _ in ordered_products), 2),
        "status": status,
        "order_date": order_date.isoformat(),
        "estimated_delivery": (order_date + datetime.timedelta(days=random.randint(3, 14))).isoformat(),
        "carrier": random.choice(carriers),
        "tracking_number": fake.bothify(text="??########").upper(),
        "shipping_address": {
            "street": fake.street_address(),
            "city": fake.city(),
            "country": customer["country"]
        },
        "return_eligible": status == "delivered" and (
            datetime.date.today() - order_date
        ).days <= 30,
        "return_reason": random.choice(return_reasons) if status == "returned" else None
    }
    orders.append(order)

print(f"✅ Generated {len(orders)} orders")


# 4. FAQ DATABASE
faqs = [
    {"id": "FAQ-001", "category": "shipping", "question": "How do I track my order?",
     "answer": "Track your order at nova.com/track using your order ID and email address."},
    {"id": "FAQ-002", "category": "returns", "question": "What is the return policy?",
     "answer": "NOVA offers free returns within 30 days of delivery for unused products in original packaging."},
    {"id": "FAQ-003", "category": "shipping", "question": "How long does shipping take?",
     "answer": "Standard: 5-7 business days. Express: 2-3 days. International: 10-14 days."},
    {"id": "FAQ-004", "category": "products", "question": "Are your products cruelty-free?",
     "answer": "Yes, all NOVA products are 100% cruelty-free and dermatologist tested."},
    {"id": "FAQ-005", "category": "products", "question": "How do I find my foundation shade?",
     "answer": "Use our Shade Finder at nova.com/shadefinder or chat with our beauty advisors."},
    {"id": "FAQ-006", "category": "orders", "question": "Can I cancel my order?",
     "answer": "Orders can be cancelled within 2 hours of placement. After that, you must request a return."},
    {"id": "FAQ-007", "category": "shipping", "question": "Do you ship internationally?",
     "answer": "Yes, NOVA ships to 14 countries. Duties and taxes may apply at checkout."},
    {"id": "FAQ-008", "category": "orders", "question": "How do I apply a promo code?",
     "answer": "Enter your promo code at checkout in the Discount Code field before completing payment."},
    {"id": "FAQ-009", "category": "products", "question": "Are products safe for sensitive skin?",
     "answer": "Many NOVA products are formulated for sensitive skin. Check the product page for skin type compatibility."},
    {"id": "FAQ-010", "category": "returns", "question": "How long does a refund take?",
     "answer": "Refunds are processed within 5-7 business days after we receive your return."},
    {"id": "FAQ-011", "category": "account", "question": "How do I reset my password?",
     "answer": "Click Forgot Password on the login page and follow the email instructions."},
    {"id": "FAQ-012", "category": "orders", "question": "What payment methods do you accept?",
     "answer": "We accept Visa, Mastercard, Amex, PayPal, Apple Pay, and Google Pay."},
]

print(f"✅ Generated {len(faqs)} FAQs")


# 5. ASSEMBLE & SAVE DATABASE
mock_db = {
    "metadata": {
        "generated_at": datetime.datetime.now().isoformat(),
        "total_products": len(products),
        "total_customers": len(customers),
        "total_orders": len(orders),
        "total_faqs": len(faqs)
    },
    "products": products,
    "customers": customers,
    "orders": orders,
    "faqs": faqs
}

with open("nova_mock_db.json", "w") as f:
    json.dump(mock_db, f, indent=2)

print(f"\n✅ nova_mock_db.json saved!")
print(f"   Products  : {len(products)}")
print(f"   Customers : {len(customers)}")
print(f"   Orders    : {len(orders)}")
print(f"   FAQs      : {len(faqs)}")
print(f"   File size : {os.path.getsize('nova_mock_db.json') / 1024:.1f} KB")

✅ Generated 60 products
✅ Generated 200 customers
✅ Generated 500 orders
✅ Generated 12 FAQs

✅ nova_mock_db.json saved!
   Products  : 60
   Customers : 200
   Orders    : 500
   FAQs      : 12
   File size : 401.1 KB


In [5]:
# Quick preview of each collection
print("=" * 55)
print("📦 SAMPLE PRODUCT")
print("=" * 55)
print(json.dumps(products[0], indent=2))

print("\n" + "=" * 55)
print("👤 SAMPLE CUSTOMER")
print("=" * 55)
print(json.dumps(customers[0], indent=2))

print("\n" + "=" * 55)
print("🛒 SAMPLE ORDER")
print("=" * 55)
print(json.dumps(orders[0], indent=2))

print("\n" + "=" * 55)
print("❓ SAMPLE FAQ")
print("=" * 55)
print(json.dumps(faqs[0], indent=2))

📦 SAMPLE PRODUCT
{
  "product_id": "SKU-1001",
  "name": "NOVA Vitamin C Serum",
  "category": "skincare",
  "price": 100.59,
  "stock": 12,
  "rating": 4.6,
  "description": "Premium vitamin c serum by NOVA. Agent every development say.",
  "ingredients": [
    "Vitamin C",
    "Retinol",
    "Tea Tree Oil",
    "Hyaluronic Acid"
  ],
  "suitable_for": [
    "normal",
    "oily",
    "combination",
    "dry"
  ],
  "size_ml": 30
}

👤 SAMPLE CUSTOMER
{
  "customer_id": "CUST-0001",
  "name": "Benjamin Beck",
  "email": "cindy14@example.net",
  "country": "UK",
  "skin_type": "combination",
  "member_since": "2023-06-22",
  "total_orders": 12,
  "total_spent": 1300.89,
  "purchase_history": [
    "SKU-1026",
    "SKU-1022",
    "SKU-1018",
    "SKU-1056"
  ],
  "loyalty_tier": "platinum",
  "preferences": [
    "footwear",
    "hair",
    "skincare",
    "apparel"
  ]
}

🛒 SAMPLE ORDER
{
  "order_id": "TRX-1001",
  "customer_id": "CUST-0083",
  "products": [
    "SKU-1022",
    "SKU-103

In [6]:
# ── Audit Logger ──────────────────────────────────────────────
# Every MCP tool call gets logged here automatically

AUDIT_LOG_PATH = "audit_log.jsonl"

def log_audit(tool_name: str, inputs: dict, output: dict, duration_ms: float):
    """Appends one audit entry to audit_log.jsonl"""
    entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "tool": tool_name,
        "inputs": inputs,
        "output": output,
        "duration_ms": round(duration_ms, 2),
        "status": "error" if "error" in output else "success"
    }
    with jsonlines.open(AUDIT_LOG_PATH, mode="a") as writer:
        writer.write(entry)
    return entry

print("✅ Audit logger ready")
print(f"   Logging to: {AUDIT_LOG_PATH}")

✅ Audit logger ready
   Logging to: audit_log.jsonl


In [7]:
import time

# Load database into memory
with open("nova_mock_db.json", "r") as f:
    db = json.load(f)

# Build lookup indexes for fast access
orders_index    = {o["order_id"]: o for o in db["orders"]}
customers_index = {c["customer_id"]: c for c in db["customers"]}
products_index  = {p["product_id"]: p for p in db["products"]}
faqs_list       = db["faqs"]

print(f"✅ Database loaded into memory")
print(f"   Orders index    : {len(orders_index)} records")
print(f"   Customers index : {len(customers_index)} records")
print(f"   Products index  : {len(products_index)} records")

# ── TOOL 1: Get Order Status ───────────────────────────────────
def get_order_status(order_id: str) -> dict:
    """Returns current status, carrier, tracking, and ETA for an order."""
    start = time.time()
    inputs = {"order_id": order_id}

    order = orders_index.get(order_id)
    if not order:
        result = {"error": f"Order {order_id} not found"}
    else:
        result = {
            "order_id": order["order_id"],
            "status": order["status"],
            "order_date": order["order_date"],
            "estimated_delivery": order["estimated_delivery"],
            "carrier": order["carrier"],
            "tracking_number": order["tracking_number"],
            "products": order["products"],
            "total_amount": order["total_amount"],
            "return_eligible": order["return_eligible"]
        }

    log_audit("get_order_status", inputs, result, (time.time() - start) * 1000)
    return result


# ── TOOL 2: Initiate Return ────────────────────────────────────
def initiate_return(order_id: str, reason: str) -> dict:
    """Creates a return request for an eligible order."""
    start = time.time()
    inputs = {"order_id": order_id, "reason": reason}

    order = orders_index.get(order_id)
    if not order:
        result = {"error": f"Order {order_id} not found"}
    elif not order["return_eligible"]:
        result = {
            "error": "Return not eligible",
            "reason": "Either order is not delivered or return window (30 days) has passed",
            "order_status": order["status"]
        }
    else:
        return_id = f"RET-{fake.bothify(text='####').upper()}"
        result = {
            "return_id": return_id,
            "order_id": order_id,
            "reason": reason,
            "status": "initiated",
            "refund_timeline": "5-7 business days after item received",
            "return_label": f"https://nova.com/returns/{return_id}",
            "message": "Return successfully initiated. Check your email for the prepaid return label."
        }

    log_audit("initiate_return", inputs, result, (time.time() - start) * 1000)
    return result


# ── TOOL 3: Get Product Info ───────────────────────────────────
def get_product_info(product_id: str) -> dict:
    """Returns full product details including ingredients, sizing, and compatibility."""
    start = time.time()
    inputs = {"product_id": product_id}

    product = products_index.get(product_id)
    if not product:
        result = {"error": f"Product {product_id} not found"}
    else:
        result = product.copy()
        result["in_stock"] = product["stock"] > 0

    log_audit("get_product_info", inputs, result, (time.time() - start) * 1000)
    return result


# ── TOOL 4: Get Customer Profile ───────────────────────────────
def get_customer_profile(customer_id: str) -> dict:
    """Returns customer profile, purchase history, skin type, and preferences."""
    start = time.time()
    inputs = {"customer_id": customer_id}

    customer = customers_index.get(customer_id)
    if not customer:
        result = {"error": f"Customer {customer_id} not found"}
    else:
        # Enrich with recent orders
        customer_orders = [
            o for o in db["orders"]
            if o["customer_id"] == customer_id
        ][-5:]  # Last 5 orders

        result = {
            **customer,
            "recent_orders": [
                {
                    "order_id": o["order_id"],
                    "status": o["status"],
                    "order_date": o["order_date"],
                    "total_amount": o["total_amount"]
                }
                for o in customer_orders
            ],
            "total_orders_count": len(customer_orders)
        }

    log_audit("get_customer_profile", inputs, result, (time.time() - start) * 1000)
    return result


# ── TOOL 5: Search FAQs ────────────────────────────────────────
def search_faqs(query: str, top_k: int = 3) -> dict:
    """Fuzzy-searches the FAQ database and returns best matches."""
    start = time.time()
    inputs = {"query": query, "top_k": top_k}

    # Build searchable strings from FAQs
    faq_questions = [f["question"] for f in faqs_list]

    # Fuzzy match using rapidfuzz
    matches = process.extract(
        query,
        faq_questions,
        scorer=fuzz.WRatio,
        limit=top_k
    )

    results = []
    for match_text, score, idx in matches:
        if score > 40:  # Minimum relevance threshold
            faq = faqs_list[idx]
            results.append({
                "faq_id": faq["id"],
                "question": faq["question"],
                "answer": faq["answer"],
                "category": faq["category"],
                "relevance_score": round(score / 100, 2)
            })

    result = {
        "query": query,
        "results": results,
        "total_found": len(results)
    }

    log_audit("search_faqs", inputs, result, (time.time() - start) * 1000)
    return result


print("✅ All 5 MCP tools defined")
print("   Tool 1 → get_order_status()")
print("   Tool 2 → initiate_return()")
print("   Tool 3 → get_product_info()")
print("   Tool 4 → get_customer_profile()")
print("   Tool 5 → search_faqs()")

✅ Database loaded into memory
   Orders index    : 500 records
   Customers index : 200 records
   Products index  : 60 records
✅ All 5 MCP tools defined
   Tool 1 → get_order_status()
   Tool 2 → initiate_return()
   Tool 3 → get_product_info()
   Tool 4 → get_customer_profile()
   Tool 5 → search_faqs()


In [8]:
print("🧪 TESTING ALL 5 TOOLS INDIVIDUALLY")
print("=" * 55)

# Pick real IDs from database for testing
sample_order_id    = orders[0]["order_id"]
sample_customer_id = customers[0]["customer_id"]
sample_product_id  = products[0]["product_id"]

# Test 1: Order Status
print("\n📦 TOOL 1 — get_order_status()")
print("-" * 40)
result1 = get_order_status(sample_order_id)
print(json.dumps(result1, indent=2))

# Test 2: Initiate Return (find a return-eligible order)
eligible_order = next(
    (o for o in orders if o["return_eligible"]), orders[0]
)
print("\n↩️  TOOL 2 — initiate_return()")
print("-" * 40)
result2 = initiate_return(eligible_order["order_id"], "wrong_size")
print(json.dumps(result2, indent=2))

# Test 3: Product Info
print("\n🔍 TOOL 3 — get_product_info()")
print("-" * 40)
result3 = get_product_info(sample_product_id)
print(json.dumps(result3, indent=2))

# Test 4: Customer Profile
print("\n👤 TOOL 4 — get_customer_profile()")
print("-" * 40)
result4 = get_customer_profile(sample_customer_id)
# Print without full purchase history to keep it readable
preview = {k: v for k, v in result4.items() if k != "purchase_history"}
print(json.dumps(preview, indent=2))

# Test 5: Search FAQs
print("\n❓ TOOL 5 — search_faqs()")
print("-" * 40)
result5 = search_faqs("how do I return my order")
print(json.dumps(result5, indent=2))

print("\n✅ All 5 tools tested successfully")

🧪 TESTING ALL 5 TOOLS INDIVIDUALLY

📦 TOOL 1 — get_order_status()
----------------------------------------
{
  "order_id": "TRX-1001",
  "status": "shipped",
  "order_date": "2026-03-27",
  "estimated_delivery": "2026-04-09",
  "carrier": "FedEx",
  "tracking_number": "XY17046226",
  "products": [
    "SKU-1022",
    "SKU-1036",
    "SKU-1037"
  ],
  "total_amount": 241.38,
  "return_eligible": false
}

↩️  TOOL 2 — initiate_return()
----------------------------------------
{
  "return_id": "RET-6484",
  "order_id": "TRX-1002",
  "reason": "wrong_size",
  "status": "initiated",
  "refund_timeline": "5-7 business days after item received",
  "return_label": "https://nova.com/returns/RET-6484",
  "message": "Return successfully initiated. Check your email for the prepaid return label."
}

🔍 TOOL 3 — get_product_info()
----------------------------------------
{
  "product_id": "SKU-1001",
  "name": "NOVA Vitamin C Serum",
  "category": "skincare",
  "price": 100.59,
  "stock": 12,
  "rati

In [9]:
# ── Compound Demo Scenario ─────────────────────────────────────
# Customer Sarah: checks order → finds delay → initiates return
# → asks for product recommendation → all logged to audit trail

print("🎬 COMPOUND DEMO SCENARIO")
print("=" * 60)
print("Customer: Sarah Mitchell (CUST-0001)")
print("Scenario: Delayed order + return + product recommendation")
print("=" * 60)

# Find a real customer and their orders
demo_customer_id = customers[0]["customer_id"]
demo_customer    = get_customer_profile(demo_customer_id)

# Find their most recent order
demo_order_id = demo_customer["recent_orders"][0]["order_id"] \
    if demo_customer.get("recent_orders") else orders[0]["order_id"]
demo_order = get_order_status(demo_order_id)

# Find a return-eligible order
return_order = next(
    (o for o in orders if o["customer_id"] == demo_customer_id
     and o["return_eligible"]),
    eligible_order
)
return_result = initiate_return(return_order["order_id"], "not_as_described")

# Get a product recommendation
rec_product_id = random.choice(
    [p["product_id"] for p in products if p["category"] == "skincare"]
)
rec_product = get_product_info(rec_product_id)

# Build context for Groq
tool_context = f"""
TOOL RESULTS FROM NOVA BACKEND:

1. Customer Profile:
   Name: {demo_customer.get('name')}
   Skin Type: {demo_customer.get('skin_type')}
   Loyalty Tier: {demo_customer.get('loyalty_tier')}
   Total Spent: ${demo_customer.get('total_spent')}

2. Order Status (#{demo_order_id}):
   Status: {demo_order.get('status')}
   Carrier: {demo_order.get('carrier')}
   Est. Delivery: {demo_order.get('estimated_delivery')}

3. Return Request:
   Return ID: {return_result.get('return_id', 'N/A')}
   Status: {return_result.get('status', return_result.get('error', 'N/A'))}
   Refund Timeline: {return_result.get('refund_timeline', 'N/A')}

4. Recommended Product:
   Name: {rec_product.get('name')}
   Category: {rec_product.get('category')}
   Price: ${rec_product.get('price')}
   Rating: {rec_product.get('rating')}/5
   Suitable For: {rec_product.get('suitable_for', 'All skin types')}
"""

# Generate final response using Groq
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": """You are NOVA AI Support. Using the tool results provided,
            write a warm, helpful response to the customer that:
            1. Addresses their order delay
            2. Confirms their return has been initiated
            3. Recommends the product
            Keep it concise, empathetic, and on-brand."""
        },
        {
            "role": "user",
            "content": f"Customer message: 'Hi, my order hasn't arrived and I want to return a previous purchase. Can you also recommend a good skincare product for my skin type?'\n\n{tool_context}"
        }
    ],
    max_tokens=400,
    temperature=0.4
)

final_response = response.choices[0].message.content

print("\n📨 CUSTOMER MESSAGE:")
print("   'Hi, my order hasn't arrived and I want to return a previous")
print("    purchase. Can you also recommend a good skincare product?'")
print("\n🔧 TOOLS CALLED (in sequence):")
print(f"   1. get_customer_profile('{demo_customer_id}')")
print(f"   2. get_order_status('{demo_order_id}')")
print(f"   3. initiate_return('{return_order['order_id']}', 'not_as_described')")
print(f"   4. get_product_info('{rec_product_id}')")
print("\n💬 FINAL AI RESPONSE:")
print("-" * 60)
print(final_response)
print("-" * 60)
print("\n✅ Compound scenario complete — 4 tools chained successfully")

🎬 COMPOUND DEMO SCENARIO
Customer: Sarah Mitchell (CUST-0001)
Scenario: Delayed order + return + product recommendation

📨 CUSTOMER MESSAGE:
   'Hi, my order hasn't arrived and I want to return a previous
    purchase. Can you also recommend a good skincare product?'

🔧 TOOLS CALLED (in sequence):
   1. get_customer_profile('CUST-0001')
   2. get_order_status('TRX-1001')
   3. initiate_return('TRX-1002', 'not_as_described')
   4. get_product_info('SKU-1001')

💬 FINAL AI RESPONSE:
------------------------------------------------------------
Hi Benjamin, 

I'm so sorry to hear that your order hasn't arrived yet. I've checked on the status, and it looks like your order (#TRX-1001) has been shipped with FedEx and is estimated to arrive by April 9th. 

Regarding your return, I'm happy to confirm that the process for Return ID RET-4941 has been initiated. You can expect a refund within 5-7 business days after we receive the item.

As a platinum loyalty tier member, I want to ensure you find 

In [10]:
print("📋 AUDIT LOG — All Tool Calls This Session")
print("=" * 60)

entries = []
with jsonlines.open(AUDIT_LOG_PATH) as reader:
    for entry in reader:
        entries.append(entry)

for i, entry in enumerate(entries, 1):
    status_icon = "✅" if entry["status"] == "success" else "❌"
    print(f"\n{status_icon} Call #{i}")
    print(f"   Tool      : {entry['tool']}")
    print(f"   Timestamp : {entry['timestamp']}")
    print(f"   Duration  : {entry['duration_ms']} ms")
    print(f"   Status    : {entry['status']}")
    print(f"   Inputs    : {json.dumps(entry['inputs'])}")

print(f"\n{'=' * 60}")
print(f"Total tool calls logged : {len(entries)}")
success = sum(1 for e in entries if e["status"] == "success")
errors  = sum(1 for e in entries if e["status"] == "error")
print(f"✅ Successful            : {success}")
print(f"❌ Errors                : {errors}")
avg_ms = sum(e["duration_ms"] for e in entries) / len(entries) if entries else 0
print(f"⚡ Avg response time     : {avg_ms:.1f} ms")

📋 AUDIT LOG — All Tool Calls This Session

✅ Call #1
   Tool      : get_order_status
   Timestamp : 2026-03-28T20:16:08.394810
   Duration  : 0.01 ms
   Status    : success
   Inputs    : {"order_id": "TRX-1001"}

✅ Call #2
   Tool      : initiate_return
   Timestamp : 2026-03-28T20:16:08.399005
   Duration  : 0.1 ms
   Status    : success
   Inputs    : {"order_id": "TRX-1002", "reason": "wrong_size"}

✅ Call #3
   Tool      : get_product_info
   Timestamp : 2026-03-28T20:16:08.399504
   Duration  : 0.01 ms
   Status    : success
   Inputs    : {"product_id": "SKU-1001"}

✅ Call #4
   Tool      : get_customer_profile
   Timestamp : 2026-03-28T20:16:08.400064
   Duration  : 0.16 ms
   Status    : success
   Inputs    : {"customer_id": "CUST-0001"}

✅ Call #5
   Tool      : search_faqs
   Timestamp : 2026-03-28T20:16:08.402539
   Duration  : 1.96 ms
   Status    : success
   Inputs    : {"query": "how do I return my order", "top_k": 3}

✅ Call #6
   Tool      : get_customer_profile
   T

In [11]:
# ── Save server.py ─────────────────────────────────────────────
server_code = '''
import json, time, datetime, jsonlines, random
from faker import Faker
from rapidfuzz import fuzz, process

fake = Faker()

with open("../nova_mock_db.json", "r") as f:
    db = json.load(f)

orders_index    = {o["order_id"]: o for o in db["orders"]}
customers_index = {c["customer_id"]: c for c in db["customers"]}
products_index  = {p["product_id"]: p for p in db["products"]}
faqs_list       = db["faqs"]

AUDIT_LOG_PATH = "../audit_log.jsonl"

def log_audit(tool_name, inputs, output, duration_ms):
    entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "tool": tool_name,
        "inputs": inputs,
        "output": output,
        "duration_ms": round(duration_ms, 2),
        "status": "error" if "error" in output else "success"
    }
    with jsonlines.open(AUDIT_LOG_PATH, mode="a") as writer:
        writer.write(entry)
    return entry

def get_order_status(order_id):
    start = time.time()
    order = orders_index.get(order_id)
    result = order if order else {"error": f"Order {order_id} not found"}
    log_audit("get_order_status", {"order_id": order_id}, result, (time.time()-start)*1000)
    return result

def initiate_return(order_id, reason):
    start = time.time()
    order = orders_index.get(order_id)
    if not order:
        result = {"error": f"Order {order_id} not found"}
    elif not order["return_eligible"]:
        result = {"error": "Return not eligible"}
    else:
        return_id = f"RET-{fake.bothify(text=\'####\').upper()}"
        result = {"return_id": return_id, "status": "initiated",
                  "refund_timeline": "5-7 business days"}
    log_audit("initiate_return", {"order_id": order_id, "reason": reason}, result, (time.time()-start)*1000)
    return result

def get_product_info(product_id):
    start = time.time()
    product = products_index.get(product_id)
    result = product if product else {"error": f"Product {product_id} not found"}
    log_audit("get_product_info", {"product_id": product_id}, result, (time.time()-start)*1000)
    return result

def get_customer_profile(customer_id):
    start = time.time()
    customer = customers_index.get(customer_id)
    result = customer if customer else {"error": f"Customer {customer_id} not found"}
    log_audit("get_customer_profile", {"customer_id": customer_id}, result, (time.time()-start)*1000)
    return result

def search_faqs(query, top_k=3):
    start = time.time()
    questions = [f["question"] for f in faqs_list]
    matches = process.extract(query, questions, scorer=fuzz.WRatio, limit=top_k)
    results = [{"faq_id": faqs_list[i]["id"], "question": faqs_list[i]["question"],
                "answer": faqs_list[i]["answer"], "score": round(s/100,2)}
               for _, s, i in matches if s > 40]
    result = {"query": query, "results": results}
    log_audit("search_faqs", {"query": query}, result, (time.time()-start)*1000)
    return result
'''

with open("task2_mcp/server.py", "w") as f:
    f.write(server_code)

print("✅ task2_mcp/server.py saved")

# ── Download files ─────────────────────────────────────────────
from google.colab import files

files.download("nova_mock_db.json")
files.download("audit_log.jsonl")
files.download("task2_mcp/server.py")

print("\n✅ Files downloaded — push to GitHub:")
print("   → nova_mock_db.json")
print("   → audit_log.jsonl")
print("   → task2_mcp/server.py")
print("   → task2_mcp_notebook.ipynb  (File > Download > .ipynb)")




✅ task2_mcp/server.py saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Files downloaded — push to GitHub:
   → nova_mock_db.json
   → audit_log.jsonl
   → task2_mcp/server.py
   → task2_mcp_notebook.ipynb  (File > Download > .ipynb)
